# 3D Motion Capture — Google Colab Runner

End-to-end pipeline that converts a monocular video into a 3D animated skeleton and exports BVH, NPZ, and JSON animation files.

**Quick start:**
1. Mount your Google Drive (Cell 2) so you can access your video.
2. Install dependencies (Cell 3) — only needed once per Colab session.
3. Set your `REPO_URL` or `PROJECT_ROOT` in Cell 4.
4. Fill in `VIDEO_PATH` and `OUTPUT_DIR` in Cell 5.
5. Run all cells (`Runtime → Run all`).

> **GPU:** Enable a GPU runtime via *Runtime → Change runtime type → T4 GPU* for faster processing.

In [ ]:
# ── Cell 2: Environment detection & Google Drive mount ──────────────────────
import os
import sys

IN_COLAB = 'google.colab' in sys.modules
print(f'Running in Colab: {IN_COLAB}')

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    print('Google Drive mounted at /content/drive')
else:
    print('Local environment detected — Drive mount skipped.')


In [ ]:
# ── Cell 3: Install dependencies ────────────────────────────────────────────
import subprocess
import importlib

def _pkg_installed(name: str) -> bool:
    """Return True if *name* is importable."""
    try:
        importlib.import_module(name)
        return True
    except ImportError:
        return False

REQUIREMENTS_PATH = None  # will be set after repo is available (Cell 4)

# Core packages that must be present regardless of requirements.txt
CORE_DEPS = [
    ('mediapipe',   'mediapipe>=0.10.9'),
    ('cv2',         'opencv-python>=4.8'),
    ('numpy',       'numpy>=1.24,<2.0'),
    ('scipy',       'scipy>=1.11'),
    ('yaml',        'PyYAML>=6.0'),
    ('tqdm',        'tqdm>=4.66'),
    ('h5py',        'h5py>=3.9'),
]

missing = [(mod, pkg) for mod, pkg in CORE_DEPS if not _pkg_installed(mod)]

if missing:
    pkgs = [pkg for _, pkg in missing]
    print(f'Installing {len(pkgs)} missing package(s): {[p.split(">")[0].split("=")[0] for _, p in missing]}')
    subprocess.check_call(
        [sys.executable, '-m', 'pip', 'install', '--quiet'] + pkgs
    )
    print('Installation complete.')
else:
    print('All core dependencies already installed.')

# Optional: install from requirements.txt once the repo path is known.
# This cell is re-run automatically at the bottom of Cell 4 if needed.


In [ ]:
# ── Cell 4: Clone / locate project ──────────────────────────────────────────
import os
import sys
import subprocess
from pathlib import Path

# ┌──────────────────────────────────────────────────────────────────────────┐
# │  Fill in ONE of these:                                                   │
# │   • REPO_URL   — GitHub HTTPS URL (used in Colab)                        │
# │   • PROJECT_ROOT — absolute local path (used outside Colab)              │
# └──────────────────────────────────────────────────────────────────────────┘
REPO_URL     = ''   # e.g. 'https://github.com/yourname/mocap3d.git'
PROJECT_ROOT = ''   # e.g. '/Users/you/Desktop/mocap'

# ── Resolve project root ────────────────────────────────────────────────────
if IN_COLAB:
    if not REPO_URL:
        raise ValueError('Set REPO_URL to your GitHub repository URL.')
    _clone_dir = Path('/content/mocap3d')
    if not _clone_dir.exists():
        print(f'Cloning {REPO_URL} ...')
        subprocess.check_call(['git', 'clone', '--depth', '1', REPO_URL, str(_clone_dir)])
        print('Clone complete.')
    else:
        print(f'Repository already present at {_clone_dir}.')
    _root = _clone_dir
else:
    if not PROJECT_ROOT:
        # Auto-detect: walk up from this notebook's directory
        _nb_dir = Path(os.path.abspath(''))
        _root = _nb_dir.parent if (_nb_dir / 'src').exists() or (_nb_dir.parent / 'src').exists() else _nb_dir
        if not (_root / 'src').exists():
            raise FileNotFoundError(
                'Cannot locate project root automatically. Set PROJECT_ROOT explicitly.'
            )
        print(f'Auto-detected project root: {_root}')
    else:
        _root = Path(PROJECT_ROOT)
        if not _root.exists():
            raise FileNotFoundError(f'PROJECT_ROOT does not exist: {_root}')

PROJECT_ROOT = str(_root)
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)
    print(f'Added to sys.path: {PROJECT_ROOT}')

# ── Install from requirements.txt if present ────────────────────────────────
_req = _root / 'requirements.txt'
if _req.exists():
    print(f'Installing from {_req} ...')
    subprocess.check_call(
        [sys.executable, '-m', 'pip', 'install', '--quiet', '-r', str(_req)]
    )
    print('Requirements installed.')

print(f'Project root: {PROJECT_ROOT}')


In [ ]:
# ── Cell 5: Configuration ────────────────────────────────────────────────────
# Edit these values before running the pipeline.

VIDEO_PATH       = ''    # e.g. '/content/drive/MyDrive/videos/dance.mp4'
                         #   or a local path like '/Users/you/Desktop/clip.mp4'
OUTPUT_DIR       = ''    # e.g. '/content/drive/MyDrive/mocap_out'
                         #   leave empty to default to a sibling of the video

USE_GPU          = True  # Set False to force CPU inference
TARGET_FPS       = 30.0  # Output animation frame rate
SMOOTHING_CUTOFF = 6.0   # Butterworth cutoff Hz (lower = smoother, higher = snappier)
EXPORT_FORMATS   = ['bvh', 'npz', 'json']  # any subset

# Optional: path to a YAML config file (overrides individual settings above
# for any key it defines).
CONFIG_FILE = ''  # e.g. 'configs/default.yaml'

# ── Validate ────────────────────────────────────────────────────────────────
from pathlib import Path

if not VIDEO_PATH:
    raise ValueError('Set VIDEO_PATH to the input video file.')

VIDEO_PATH = Path(VIDEO_PATH)
if not VIDEO_PATH.exists():
    raise FileNotFoundError(f'Video not found: {VIDEO_PATH}')

if not OUTPUT_DIR:
    OUTPUT_DIR = VIDEO_PATH.parent / (VIDEO_PATH.stem + '_mocap_out')

OUTPUT_DIR = Path(OUTPUT_DIR)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f'Video      : {VIDEO_PATH}')
print(f'Output dir : {OUTPUT_DIR}')
print(f'GPU        : {USE_GPU}')
print(f'Target FPS : {TARGET_FPS}')
print(f'Cutoff Hz  : {SMOOTHING_CUTOFF}')
print(f'Formats    : {EXPORT_FORMATS}')


In [ ]:
# ── Cell 6: Run pipeline ─────────────────────────────────────────────────────
import time
from src.config.settings import load_settings
from src.pipeline import MocapPipeline

# Load settings (from file if provided, else defaults)
settings = load_settings(CONFIG_FILE if CONFIG_FILE else None)

# Apply notebook-level overrides
settings.video.target_fps          = TARGET_FPS
settings.filtering.butterworth_cutoff = SMOOTHING_CUTOFF
settings.animation.export_formats  = EXPORT_FORMATS
settings.animation.output_dir      = str(OUTPUT_DIR)

if USE_GPU:
    settings.device = 'auto'  # selects cuda > mps > cpu
else:
    settings.device = 'cpu'

# Run
pipeline = MocapPipeline(settings)

print(f'Starting pipeline on: {VIDEO_PATH.name}')
t0 = time.perf_counter()

result = pipeline.process(video_path=VIDEO_PATH, output_dir=OUTPUT_DIR)

elapsed = time.perf_counter() - t0
print(f'\nPipeline finished in {elapsed:.1f}s')


In [ ]:
# ── Cell 7: View results ─────────────────────────────────────────────────────
from IPython.display import display, Video, HTML

# ── Summary ─────────────────────────────────────────────────────────────────
print('=== Processing Summary ===')
print(f'  Source         : {result.source_video.name}')
print(f'  Frames         : {result.frame_count}')
print(f'  Detection rate : {result.detection_rate * 100:.1f}%')
print(f'  Elapsed        : {result.processing_time_sec:.2f}s')
if result.frame_count and result.processing_time_sec:
    print(f'  Throughput     : {result.frame_count / result.processing_time_sec:.1f} fps')

if result.output_files:
    print('\n  Output files:')
    for fmt, path in sorted(result.output_files.items()):
        size_kb = path.stat().st_size / 1024 if path.exists() else 0
        print(f'    [{fmt:>5}]  {path.name}  ({size_kb:.1f} KB)')

if result.metrics is not None:
    print(f'\n  Metrics: {result.metrics}')

# ── Preview video ────────────────────────────────────────────────────────────
_preview = result.output_files.get('preview')
if _preview is None:
    # Try common naming convention
    _candidate = OUTPUT_DIR / (VIDEO_PATH.stem + '_preview.mp4')
    _preview = _candidate if _candidate.exists() else None

if _preview and _preview.exists():
    print(f'\nPreview: {_preview}')
    display(Video(str(_preview), embed=True, width=640))
else:
    print('\nNo preview video available.')


In [ ]:
# ── Cell 8: Download results ─────────────────────────────────────────────────
if IN_COLAB:
    from google.colab import files as colab_files

    if not result.output_files:
        print('No output files to download.')
    else:
        print('Downloading output files...')
        for fmt, path in sorted(result.output_files.items()):
            if path.exists():
                print(f'  Downloading {path.name} ...')
                colab_files.download(str(path))
            else:
                print(f'  Skipping {path.name} (file not found)')
        print('Download complete.')
else:
    print('Not running in Colab — files are already at:')
    for fmt, path in sorted(result.output_files.items()):
        print(f'  {path}')
